# Atlassian Rovo A2A DCR OAuth Walkthrough

This notebook verifies the OAuth/DCR theory for `KAN-5` using Atlassian Rovo A2A as the concrete example.

It is intentionally safe by default:

1. Public discovery and metadata cells run without credentials.
2. Dynamic Client Registration is a real external write, so it only runs when `A2A_ROVO_DCR_EXECUTE=1`.
3. The interactive localhost callback flow starts a temporary server on `localhost:3000`, opens the consent URL in a browser, exchanges the callback code, and prints the bearer and refresh token for local manual testing.
4. Live A2A message sending uses the token captured by the interactive flow, or `ATLASSIAN_A2A_ACCESS_TOKEN` when supplied.

Do not paste access tokens, refresh tokens, authorization codes, client secrets, state payloads, or PKCE verifiers into chat, issues, docs, or committed notebook output. A DCR `client_id` is a public identifier and can be used to build the consent URL; any returned `client_secret` must still be treated as a secret.

## Verified Public Inputs

Atlassian's public docs describe this expected flow:

1. Fetch the public Agent Card at `https://a2a.atlassian.com/.well-known/agent.json`.
2. Read OAuth authorization-code settings and required scopes from the card.
3. Register the client through Dynamic Client Registration.
4. Redirect the user to Atlassian consent with PKCE `S256`.
5. Exchange the authorization code for access and refresh tokens.
6. Send A2A requests to `https://a2a.atlassian.com/v1/rovo` with `Authorization: Bearer <token>`.
7. Use the refresh token for later access-token renewal.

References:

- https://developer.atlassian.com/cloud/rovo-a2a/authentication/
- https://developer.atlassian.com/cloud/rovo-a2a/getting-started/
- https://developer.atlassian.com/cloud/rovo-a2a/end-to-end-testing/
- https://support.atlassian.com/atlassian-ai-gateway/docs/set-up-agent2agent-connections/

For the InnomightLabs `agent2agent_client` skill, set `registry_url` to the Agent Card URL: `https://a2a.atlassian.com/.well-known/agent.json`. Do not use the JSON-RPC service URL `https://a2a.atlassian.com/v1/rovo` as the registry/discovery URL; that endpoint only accepts authenticated `POST` requests.

## Prerequisites

Run the notebook with a Python environment that has `httpx` installed.

Recommended local launch:

```bash
cd api
uv run jupyter lab ../developer-manual/atlassian-rovo-a2a-dcr-oauth.ipynb
```

Optional environment variables:

```bash
export A2A_ROVO_AGENT_CARD_URL='https://a2a.atlassian.com/.well-known/agent.json'
export A2A_ROVO_PRM_URL='https://a2a.atlassian.com/.well-known/oauth-protected-resource/v1/rovo'
export A2A_LOCAL_CALLBACK_BASE_URL='http://localhost:3000'
export A2A_ROVO_CALLBACK_PATH='/oauth/callback'
export ATLASSIAN_SITE_URL='https://vslala007.atlassian.net'
export A2A_ROVO_DCR_EXECUTE='0'
export A2A_ROVO_CLIENT_ID='<client-id-from-prior-dcr-response>'
export ATLASSIAN_A2A_ACCESS_TOKEN='<access-token-for-local-testing-only>'
```

`A2A_LOCAL_CALLBACK_BASE_URL` and `A2A_ROVO_CALLBACK_PATH` control the notebook redirect URI. The default is `http://localhost:3000/oauth/callback`.

When installing the `agent2agent_client` skill manually, use `Registry Set Name = Atlassian Rovo` and `Registry URL = https://a2a.atlassian.com/.well-known/agent.json`.

`ATLASSIAN_SITE_URL` is test context for prompts and org troubleshooting. It is not added directly to the Atlassian `/authorize` URL; Atlassian resolves the user's site/org after the user signs in and the A2A request is executed.

In [32]:
import base64
import hashlib
import threading
import time
import webbrowser
from http.server import BaseHTTPRequestHandler, HTTPServer
import json
import os
import secrets
from pprint import pprint
from urllib.parse import parse_qs, urlencode, urlparse
from uuid import uuid4

import httpx

AGENT_CARD_URL = os.getenv("A2A_ROVO_AGENT_CARD_URL", "https://a2a.atlassian.com/.well-known/agent.json")
PRM_URL = os.getenv("A2A_ROVO_PRM_URL", "https://a2a.atlassian.com/.well-known/oauth-protected-resource/v1/rovo")
LOCAL_CALLBACK_BASE_URL = os.getenv("A2A_LOCAL_CALLBACK_BASE_URL", "http://localhost:3000").rstrip("/")
CALLBACK_PATH = os.getenv("A2A_ROVO_CALLBACK_PATH", "/oauth/callback")
if not CALLBACK_PATH.startswith("/"):
    CALLBACK_PATH = f"/{CALLBACK_PATH}"
REDIRECT_URI = os.getenv("A2A_ROVO_REDIRECT_URI", f"{LOCAL_CALLBACK_BASE_URL}{CALLBACK_PATH}")
ATLASSIAN_SITE_URL = os.getenv("ATLASSIAN_SITE_URL", "https://vslala007.atlassian.net")
DCR_EXECUTE = os.getenv("A2A_ROVO_DCR_EXECUTE", "0") == "1"

print("AGENT_CARD_URL:", AGENT_CARD_URL)
print("PRM_URL:", PRM_URL)
print("REDIRECT_URI:", REDIRECT_URI)
print("ATLASSIAN_SITE_URL:", ATLASSIAN_SITE_URL)
print("DCR_EXECUTE:", DCR_EXECUTE)

AGENT_CARD_URL: https://a2a.atlassian.com/.well-known/agent.json
PRM_URL: https://a2a.atlassian.com/.well-known/oauth-protected-resource/v1/rovo
REDIRECT_URI: http://localhost:3000/oauth/callback
ATLASSIAN_SITE_URL: https://vslala007.atlassian.net
DCR_EXECUTE: False


In [33]:
def fetch_json(url: str) -> dict:
    with httpx.Client(timeout=30.0, follow_redirects=False) as client:
        response = client.get(url, headers={"Accept": "application/json"})
        response.raise_for_status()
        return response.json()


agent_card = fetch_json(AGENT_CARD_URL)
protected_resource_metadata = fetch_json(PRM_URL)

print("Agent Card summary:")
pprint({
    "name": agent_card.get("name"),
    "protocolVersion": agent_card.get("protocolVersion"),
    "url": agent_card.get("url"),
    "preferredTransport": agent_card.get("preferredTransport"),
    "streaming": (agent_card.get("capabilities") or {}).get("streaming"),
    "security_keys": list((agent_card.get("securitySchemes") or {}).keys()),
})

print("\nProtected resource metadata:")
pprint(protected_resource_metadata)

Agent Card summary:
{'name': 'Atlassian Rovo',
 'preferredTransport': 'JSONRPC',
 'protocolVersion': '0.3.0',
 'security_keys': ['oauth2'],
 'streaming': True,
 'url': 'https://a2a.atlassian.com/v1/rovo'}

Protected resource metadata:
{'authorization_servers': ['https://auth.atlassian.com/rIh7rYbkJiQdDle1kZAShJjyflKVUdtV'],
 'bearer_methods_supported': ['header'],
 'resource': 'https://a2a.atlassian.com',
 'resource_documentation': 'https://www.atlassian.com/platform/remote-mcp-server',
 'scopes_supported': ['read:me', 'offline_access', 'full_access:chat:rovo']}


In [34]:
def origin(url: str) -> str:
    parsed = urlparse(url)
    return f"{parsed.scheme}://{parsed.netloc}" if parsed.scheme and parsed.netloc else ""


def authorization_code_flow(card: dict) -> tuple[str, dict]:
    for name, scheme in (card.get("securitySchemes") or {}).items():
        oauth2 = scheme.get("oauth2SecurityScheme") if isinstance(scheme.get("oauth2SecurityScheme"), dict) else scheme
        flows = oauth2.get("flows") if isinstance(oauth2, dict) else None
        flow = (flows or {}).get("authorizationCode") or (flows or {}).get("authorization_code")
        if flow:
            return name, flow
    raise AssertionError("No OAuth authorization-code flow found in Agent Card")


def required_scopes(card: dict, scheme_name: str, flow: dict) -> list[str]:
    scopes = []
    for requirement in card.get("security") or card.get("securityRequirements") or []:
        values = requirement.get(scheme_name)
        if isinstance(values, list):
            scopes.extend(str(value) for value in values)
    if not scopes:
        scopes.extend(str(key) for key in (flow.get("scopes") or {}).keys())
    return list(dict.fromkeys(scope for scope in scopes if scope))


def dcr_target_from_card(card: dict) -> str | None:
    for extension in (card.get("capabilities") or {}).get("extensions") or []:
        params = extension.get("params") or {}
        target_url = params.get("target_url") or params.get("targetUrl")
        if target_url and "/dcr/" in target_url:
            return target_url
    return None


def auth_server_metadata_url(auth_server: str) -> str:
    return f"{auth_server.rstrip('/')}/.well-known/oauth-authorization-server"


scheme_name, auth_code_flow = authorization_code_flow(agent_card)
scopes = required_scopes(agent_card, scheme_name, auth_code_flow)
authorization_url = auth_code_flow.get("authorizationUrl") or auth_code_flow.get("authorization_url")
token_url = auth_code_flow.get("tokenUrl") or auth_code_flow.get("token_url")
refresh_url = auth_code_flow.get("refreshUrl") or auth_code_flow.get("refresh_url") or token_url

auth_server_urls = protected_resource_metadata.get("authorization_servers") or []
authorization_server_metadata = fetch_json(auth_server_metadata_url(auth_server_urls[0])) if auth_server_urls else {}
registration_endpoint = (
    authorization_server_metadata.get("registration_endpoint")
    or dcr_target_from_card(agent_card)
)

derived = {
    "scheme_name": scheme_name,
    "service_url": agent_card.get("url"),
    "authorization_url": authorization_url,
    "token_url": token_url,
    "refresh_url": refresh_url,
    "scopes": scopes,
    "authorization_servers": auth_server_urls,
    "registration_endpoint": registration_endpoint,
    "service_origin": origin(agent_card.get("url", "")),
    "auth_origin": origin(authorization_url or ""),
}
pprint(derived)

{'auth_origin': 'https://auth.atlassian.com',
 'authorization_servers': ['https://auth.atlassian.com/rIh7rYbkJiQdDle1kZAShJjyflKVUdtV'],
 'authorization_url': 'https://auth.atlassian.com/authorize',
 'refresh_url': 'https://auth.atlassian.com/oauth/token',
 'registration_endpoint': 'https://auth.atlassian.com/rIh7rYbkJiQdDle1kZAShJjyflKVUdtV/dcr/register',
 'scheme_name': 'oauth2',
 'scopes': ['read:me', 'offline_access', 'full_access:chat:rovo'],
 'service_origin': 'https://a2a.atlassian.com',
 'service_url': 'https://a2a.atlassian.com/v1/rovo',
 'token_url': 'https://auth.atlassian.com/oauth/token'}


In [35]:
required = {"read:me", "offline_access", "full_access:chat:rovo"}

assert agent_card.get("url") == "https://a2a.atlassian.com/v1/rovo"
assert scheme_name == "oauth2"
assert authorization_url == "https://auth.atlassian.com/authorize"
assert token_url == "https://auth.atlassian.com/oauth/token"
assert required.issubset(set(scopes)), scopes
assert "S256" in (authorization_server_metadata.get("code_challenge_methods_supported") or ["S256"])
assert "authorization_code" in (authorization_server_metadata.get("grant_types_supported") or ["authorization_code"])
assert "refresh_token" in (authorization_server_metadata.get("grant_types_supported") or ["refresh_token"])
assert registration_endpoint and registration_endpoint.endswith("/dcr/register")
assert origin(agent_card["url"]) != origin(authorization_url)

print("Theory verified from live discovery:")
print("- Agent Card is public")
print("- Rovo A2A service URL is OAuth protected")
print("- OAuth endpoints are cross-origin and metadata-bound")
print("- DCR endpoint is discoverable")
print("- Required scopes include offline_access for refresh tokens")
print("- PKCE S256 is supported and should be mandatory")

Theory verified from live discovery:
- Agent Card is public
- Rovo A2A service URL is OAuth protected
- OAuth endpoints are cross-origin and metadata-bound
- DCR endpoint is discoverable
- Required scopes include offline_access for refresh tokens
- PKCE S256 is supported and should be mandatory


In [36]:
dcr_payload = {
    "client_name": "InnomightLabs Agent2Agent Client Local Test",
    "redirect_uris": [REDIRECT_URI],
    "grant_types": ["authorization_code", "refresh_token"],
    "response_types": ["code"],
    "scope": " ".join(scopes),
    "token_endpoint_auth_method": "none",
}

print("DCR registration endpoint:", registration_endpoint)
print("DCR payload that production code should generate:")
pprint(dcr_payload)

DCR registration endpoint: https://auth.atlassian.com/rIh7rYbkJiQdDle1kZAShJjyflKVUdtV/dcr/register
DCR payload that production code should generate:
{'client_name': 'InnomightLabs Agent2Agent Client Local Test',
 'grant_types': ['authorization_code', 'refresh_token'],
 'redirect_uris': ['http://localhost:3000/oauth/callback'],
 'response_types': ['code'],
 'scope': 'read:me offline_access full_access:chat:rovo',
 'token_endpoint_auth_method': 'none'}


In [37]:
client_registration = None

if not DCR_EXECUTE:
    print("DRY RUN: set A2A_ROVO_DCR_EXECUTE=1 to create a real Atlassian DCR client.")
    print("If you already have a DCR client id, set A2A_ROVO_CLIENT_ID before running the consent URL cell.")
else:
    with httpx.Client(timeout=30.0, follow_redirects=False) as client:
        response = client.post(
            registration_endpoint,
            headers={"Content-Type": "application/json", "Accept": "application/json"},
            json=dcr_payload,
        )
    print("DCR status:", response.status_code)
    if response.status_code >= 400:
        print(response.text[:2000])
        response.raise_for_status()
    client_registration = response.json()
    if client_registration.get("token_endpoint_auth_method") == "none":
        print("DCR registered a public PKCE client. Store client_id; do not require client_secret for token exchange.")
    redacted = {
        key: ("<redacted>" if "secret" in key or "token" in key else value)
        for key, value in client_registration.items()
    }
    print("DCR response, redacted:")
    pprint(redacted)

DRY RUN: set A2A_ROVO_DCR_EXECUTE=1 to create a real Atlassian DCR client.
If you already have a DCR client id, set A2A_ROVO_CLIENT_ID before running the consent URL cell.


In [38]:
def pkce_pair() -> tuple[str, str]:
    verifier = secrets.token_urlsafe(64)
    digest = hashlib.sha256(verifier.encode("ascii")).digest()
    challenge = base64.urlsafe_b64encode(digest).rstrip(b"=").decode("ascii")
    return verifier, challenge


code_verifier, code_challenge = pkce_pair()
state = secrets.token_urlsafe(32)
client_id = ((client_registration or {}).get("client_id") or os.getenv("A2A_ROVO_CLIENT_ID", "")).strip()

if not client_id:
    consent_url = None
    print("No real client_id is available yet, so no clickable consent URL will be generated.")
    print("Run the DCR cell with A2A_ROVO_DCR_EXECUTE=1, or provide A2A_ROVO_CLIENT_ID from a prior DCR response.")
    print("DCR endpoint:", registration_endpoint)
    print("Redirect URI to register:", REDIRECT_URI)
else:
    assert not client_id.startswith("<"), "client_id must come from DCR; placeholders are not valid"
    authorization_params = {
        "client_id": client_id,
        "redirect_uri": REDIRECT_URI,
        "response_type": "code",
        "scope": " ".join(scopes),
        "state": state,
        "code_challenge": code_challenge,
        "code_challenge_method": "S256",
        "prompt": "consent",
    }
    consent_url = f"{authorization_url}?{urlencode(authorization_params)}"
    print("Consent URL:")
    print(consent_url)
    print("\nConsent URL includes: client_id, response_type, redirect_uri, scope, state, code_challenge, code_challenge_method, prompt=consent")
    print("\nNotebook test note: this URL uses state and PKCE generated in this kernel.")
    print("Production must generate the URL from persisted backend state before the callback can complete automatically.")

print("\nProduction note: store state and code_verifier encrypted, single-use, and user-bound. Do not expose them to the LLM.")

No real client_id is available yet, so no clickable consent URL will be generated.
Run the DCR cell with A2A_ROVO_DCR_EXECUTE=1, or provide A2A_ROVO_CLIENT_ID from a prior DCR response.
DCR endpoint: https://auth.atlassian.com/rIh7rYbkJiQdDle1kZAShJjyflKVUdtV/dcr/register
Redirect URI to register: http://localhost:3000/oauth/callback

Production note: store state and code_verifier encrypted, single-use, and user-bound. Do not expose them to the LLM.


## Interactive Localhost Callback Flow

This cell starts a temporary HTTP server on the redirect URI host/port, opens the Atlassian consent URL in your browser, waits for the callback, exchanges the authorization code, and stores the token response in notebook variables.

The cell prints the bearer token and refresh token because this notebook is a local manual verification tool. Do not save committed notebook outputs after running it.

In [39]:
def run_local_oauth_flow(timeout_seconds: int = 600) -> dict:
    if not consent_url:
        raise RuntimeError("No consent_url available. Run DCR first or set A2A_ROVO_CLIENT_ID.")

    parsed_redirect = urlparse(REDIRECT_URI)
    if parsed_redirect.scheme != "http":
        raise RuntimeError("This notebook callback server only supports http://localhost redirects.")
    if parsed_redirect.hostname not in {"localhost", "127.0.0.1"}:
        raise RuntimeError(f"Refusing to bind non-local callback host: {parsed_redirect.hostname}")

    callback_host = "127.0.0.1"
    callback_port = parsed_redirect.port or 80
    callback_path = parsed_redirect.path or "/"
    callback_result: dict[str, dict[str, str]] = {}

    class OAuthCallbackHandler(BaseHTTPRequestHandler):
        def do_GET(self):
            request_url = urlparse(self.path)
            if request_url.path != callback_path:
                self.send_response(404)
                self.end_headers()
                self.wfile.write(b"Not found")
                return

            callback_result["params"] = {
                key: values[0]
                for key, values in parse_qs(request_url.query, keep_blank_values=True).items()
            }
            self.send_response(200)
            self.send_header("Content-Type", "text/html; charset=utf-8")
            self.end_headers()
            self.wfile.write(
                b"<html><body><h1>Atlassian OAuth callback received</h1>"
                b"<p>You can return to the notebook.</p></body></html>"
            )

        def log_message(self, format, *args):
            return

    server = HTTPServer((callback_host, callback_port), OAuthCallbackHandler)
    server.timeout = timeout_seconds
    thread = threading.Thread(target=server.handle_request, daemon=True)
    thread.start()

    print(f"Listening for OAuth callback on {REDIRECT_URI}")
    print("Opening Atlassian consent URL in the browser...")
    opened = webbrowser.open(consent_url)
    if not opened:
        print("Browser launch was not confirmed. Open this URL manually:")
        print(consent_url)

    thread.join(timeout_seconds + 5)
    server.server_close()

    params = callback_result.get("params")
    if not params:
        raise TimeoutError(f"No OAuth callback received within {timeout_seconds} seconds")
    if params.get("error"):
        raise RuntimeError(f"OAuth callback error: {params.get('error_description') or params.get('error')}")
    if params.get("state") != state:
        raise RuntimeError("Returned OAuth state does not match the generated state")

    token_data = {
        "grant_type": "authorization_code",
        "code": params["code"],
        "redirect_uri": REDIRECT_URI,
        "client_id": client_id,
        "code_verifier": code_verifier,
    }
    token_auth_method = (client_registration or {}).get("token_endpoint_auth_method") or "none"
    client_secret = (client_registration or {}).get("client_secret") or os.getenv("A2A_ROVO_CLIENT_SECRET", "")
    basic_auth = None
    if token_auth_method == "client_secret_post" and client_secret:
        token_data["client_secret"] = client_secret
    elif token_auth_method == "client_secret_basic" and client_secret:
        basic_auth = (client_id, client_secret)

    with httpx.Client(timeout=30.0, follow_redirects=False) as client:
        response = client.post(
            token_url,
            headers={"Content-Type": "application/x-www-form-urlencoded", "Accept": "application/json"},
            data=token_data,
            auth=basic_auth,
        )
    print("Token exchange status:", response.status_code)
    if response.status_code >= 400:
        print(response.text[:2000])
        response.raise_for_status()
    return response.json()


token_response = run_local_oauth_flow()
bearer_token = token_response["access_token"]
refresh_token = token_response.get("refresh_token")

print("Bearer token:")
print(f"Bearer {bearer_token}")
print("Refresh token:")
print(refresh_token or "<not returned>")

RuntimeError: No consent_url available. Run DCR first or set A2A_ROVO_CLIENT_ID.

## Production Callback and Manual Exchange Fallback

After the browser redirects to `/skills/agent2agent_client/oauth/callback`, production code should:

1. Validate and atomically consume the persisted `state`.
2. Load the encrypted PKCE `code_verifier`.
3. Load the saved DCR client registration.
4. Exchange `code` at the card-derived `tokenUrl`.
5. Persist `access_token`, `refresh_token`, `token_type`, `expires_in`, granted scopes, client id, provider key, target origin, and owner user using encryption.
6. Mark any pending A2A call resumable.

The interactive localhost cell above is the preferred notebook flow. The following fallback cell is useful only if you manually captured a callback URL and set `A2A_ROVO_AUTHORIZATION_RESPONSE_URL`.

In [ ]:
authorization_response_url = os.getenv("A2A_ROVO_AUTHORIZATION_RESPONSE_URL", "")

if not authorization_response_url:
    print("No A2A_ROVO_AUTHORIZATION_RESPONSE_URL supplied; skipping code exchange.")
else:
    parsed = urlparse(authorization_response_url)
    params = parse_qs(parsed.query)
    code = (params.get("code") or [""])[0]
    returned_state = (params.get("state") or [""])[0]
    assert code, "Callback URL did not include code"
    assert returned_state, "Callback URL did not include state"
    assert returned_state == state, "Returned OAuth state does not match the generated notebook state"

    token_request = {
        "grant_type": "authorization_code",
        "code": "<redacted>",
        "redirect_uri": REDIRECT_URI,
        "client_id": client_id,
        "code_verifier": "<redacted>",
    }
    print("Token exchange request shape:")
    pprint(token_request)

    with httpx.Client(timeout=30.0, follow_redirects=False) as client:
        response = client.post(
            token_url,
            headers={"Content-Type": "application/x-www-form-urlencoded", "Accept": "application/json"},
            data={
                "grant_type": "authorization_code",
                "code": code,
                "redirect_uri": REDIRECT_URI,
                "client_id": client_id,
                "code_verifier": code_verifier,
            },
        )
    print("Token exchange status:", response.status_code)
    if response.status_code >= 400:
        print(response.text[:2000])
        response.raise_for_status()
    token_response = response.json()
    bearer_token = token_response["access_token"]
    refresh_token = token_response.get("refresh_token")
    print("Bearer token:")
    print(f"Bearer {bearer_token}")
    print("Refresh token:")
    print(refresh_token or "<not returned>")

In [ ]:
refresh_request_shape = {
    "grant_type": "refresh_token",
    "client_id": client_id,
    "refresh_token": "<encrypted refresh token from grant store>",
}

print("Refresh request endpoint:", refresh_url)
print("Refresh request shape:")
pprint(refresh_request_shape)
print("Production note: persist rotated access_token and refresh_token atomically, then retry the A2A request once.")

In [ ]:
access_token = globals().get("bearer_token") or os.getenv("ATLASSIAN_A2A_ACCESS_TOKEN", "")

payload = {
    "jsonrpc": "2.0",
    "id": str(uuid4()),
    "method": "message/stream",
    "params": {
        "message": {
            "kind": "message",
            "messageId": str(uuid4()),
            "role": "user",
            "parts": [
                {
                    "kind": "text",
                    "text": f"Hello world from the InnomightLabs A2A OAuth notebook. Use the Atlassian site {ATLASSIAN_SITE_URL}.",
                }
            ],
        }
    },
}

if not access_token:
    print("No token captured by the OAuth flow and no ATLASSIAN_A2A_ACCESS_TOKEN supplied; skipping live A2A request.")
    print("Request shape:")
    pprint({"url": agent_card.get("url"), "headers": {"Authorization": "Bearer <redacted>"}, "payload": payload})
else:
    with httpx.Client(timeout=240.0, follow_redirects=False) as client:
        response = client.post(
            agent_card["url"],
            headers={
                "Content-Type": "application/json",
                "Accept": "text/event-stream",
                "Authorization": f"Bearer {access_token}",
            },
            json=payload,
        )
    print("A2A status:", response.status_code)
    print(response.text[:4000])

## Mapping Back to InnomightLabs

This notebook proves the implementation plan needs these production behaviors:

- `agent2agent_client` install can accept Rovo's Agent Card URL and return an authorization URL after DCR.
- DCR client registration must be stored per owning user, installed skill, and provider key.
- OAuth endpoint validation must allow cross-origin endpoints when they are metadata-bound: `a2a.atlassian.com` delegates OAuth to `auth.atlassian.com`.
- PKCE verifier and state must be persisted encrypted, expiring, user-bound, and single-use.
- `send_message` should create a pending call when OAuth interrupts execution.
- Callback should store encrypted grants and mark the pending call resumable.
- `resume_message(pending_call_id)` should replay the original call without exposing credentials.
- Refresh should use `offline_access`, handle token rotation, and retry the original request once.
- `403` JSON-RPC codes `-32008` and `-32007` should be surfaced as organization/A2A enablement problems, not OAuth login failures.